# Paper III: three-seed 50-round reconstruction at clipping threshold 3
Illustrative reconstructed experiment. Nonpublic reference preprocessing prevents end-to-end DP certification. Three seeds, C1/C2 and D0/D0C/D2.

In [ ]:
%pip install -q opacus xlrd
import opacus, xlrd
print('Opacus:',opacus.__version__)


In [ ]:
# Reconstructed federated DP-SGD validation: single seed, 50 rounds
import numpy as np,pandas as pd
from scipy.special import expit
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score,roc_curve,brier_score_loss
from sklearn.isotonic import IsotonicRegression
from opacus.accountants import RDPAccountant
SEED=17286;ROUNDS=50;LOCAL_STEPS=2;NCLIENTS=5;Q=.05;LR=.15;CLIP=3.0;DELTA=1e-5
url='https://archive.ics.uci.edu/ml/machine-learning-databases/00350/default%20of%20credit%20card%20clients.xls'
try: df=pd.read_excel(url,header=1)
except Exception:
 from google.colab import files
 print('Upload the original UCI credit card default spreadsheet')
 df=pd.read_excel(next(iter(files.upload())),header=1)
df.columns=[str(c).strip() for c in df.columns]
X=df.drop(columns=['ID','default payment next month'],errors='ignore').to_numpy(float)
y=df['default payment next month'].to_numpy(int)
assert len(y)==30000
all_idx=np.arange(len(y))
trainpool,test=train_test_split(all_idx,test_size=.20,stratify=y,random_state=SEED)
train,hold=train_test_split(trainpool,test_size=.25,stratify=y[trainpool],random_state=SEED+1)
train,cal=train_test_split(train,test_size=.20,stratify=y[train],random_state=SEED+2)
reference,hold=train_test_split(hold,test_size=.50,stratify=y[hold],random_state=SEED+3)
scaler=StandardScaler().fit(X[reference]);z=np.column_stack([np.ones(len(y)),scaler.transform(X)])
def partitions(cond):
 rng=np.random.default_rng(SEED+100+(cond=='C2'))
 if cond=='C1':
  ids=rng.permutation(train);return [a for a in np.array_split(ids,NCLIENTS)]
 # label skew with equal-ish client sizes, label proportions vary
 z0=rng.permutation(train[y[train]==0]);z1=rng.permutation(train[y[train]==1])
 cuts0=np.cumsum([int(len(z0)*v) for v in [.10,.15,.20,.25]])
 cuts1=np.cumsum([int(len(z1)*v) for v in [.40,.25,.15,.10]])
 return [np.r_[a,b] for a,b in zip(np.split(z0,cuts0),np.split(z1,cuts1))]
def train_fed(cond,treatment):
 rng=np.random.default_rng(SEED+500+{'D0':0,'D0C':1,'D1':2,'D2':3,'D3':4}[treatment])
 clients=partitions(cond);weights=np.array([len(a) for a in clients],float);weights/=weights.sum()
 w=np.zeros(z.shape[1]);sigma={'D0':0.,'D0C':0.,'D1':.8,'D2':1.5,'D3':2.5}[treatment]
 clip_active=0;total_examples=0
 for rnd in range(ROUNDS):
  local=[]
  for ids in clients:
   wc=w.copy();zi=z[ids];yi=y[ids]
   for step in range(LOCAL_STEPS):
    mask=rng.random(len(ids))<Q
    if not mask.any():continue
    xx=zi[mask];yy=yi[mask]
    per=(expit(xx@wc)-yy)[:,None]*xx
    norms=np.linalg.norm(per,axis=1)
    total_examples+=len(norms)
    if treatment!='D0':
     clip_active+=int(np.sum(norms>CLIP))
     per*=np.minimum(1,CLIP/(norms+1e-12))[:,None]
    # Poisson subsampling, sum over examples / expected batch size.
    grad=per.sum(axis=0)/(Q*len(ids))
    if sigma>0:
     grad+=rng.normal(0,sigma*CLIP/(Q*len(ids)),size=len(w))
    wc-=LR*grad
   local.append(wc)
  w=np.average(local,axis=0,weights=weights)
 accountant=None
 if sigma>0:
  acc=RDPAccountant()
  for _ in range(ROUNDS*LOCAL_STEPS):acc.step(noise_multiplier=sigma,sample_rate=Q)
  accountant=acc.get_epsilon(delta=DELTA)
 return w,accountant,clip_active/max(total_examples,1),[len(a) for a in clients]
def logloss(prob,yy):
 p=np.clip(prob,1e-8,1-1e-8);return -(yy*np.log(p)+(1-yy)*np.log1p(-p))
def attack(score,labels):
 fpr,tpr,_=roc_curve(labels,score)
 return roc_auc_score(labels,score),float(max(tpr[fpr<=.01]))
# independent reference model is trained on data disjoint from target training
# It is not a LiRA shadow-model ensemble.
def fit_reference():
 w=np.zeros(z.shape[1]);ids=reference
 for _ in range(300):
  g=z[ids].T@(expit(z[ids]@w)-y[ids])/len(ids);w-=.15*g
 return w
wr=fit_reference()
rng=np.random.default_rng(SEED+999)
m=[];n=[]
for cls in [0,1]:
 mi=train[y[train]==cls];ni=hold[y[hold]==cls];k=min(len(mi),len(ni))
 m.extend(rng.choice(mi,k,replace=False));n.extend(rng.choice(ni,k,replace=False))
cand=np.r_[m,n];labels=np.r_[np.ones(len(m),int),np.zeros(len(n),int)]
assert np.array_equal(np.bincount(y[m]),np.bincount(y[n]))
rows=[]
for seed in [17286,17287,17288]:
 SEED=seed
 # Reconstruct disjoint splits and preprocessing independently for each seed.
 trainpool,test=train_test_split(all_idx,test_size=.20,stratify=y,random_state=SEED)
 train,hold=train_test_split(trainpool,test_size=.25,stratify=y[trainpool],random_state=SEED+1)
 train,cal=train_test_split(train,test_size=.20,stratify=y[train],random_state=SEED+2)
 reference,hold=train_test_split(hold,test_size=.50,stratify=y[hold],random_state=SEED+3)
 scaler=StandardScaler().fit(X[reference])
 z=np.column_stack([np.ones(len(y)),scaler.transform(X)])
 wr=fit_reference()
 rng=np.random.default_rng(SEED+999)
 m=[];n=[]
 for cls in [0,1]:
  mi=train[y[train]==cls];ni=hold[y[hold]==cls];k=min(len(mi),len(ni))
  m.extend(rng.choice(mi,k,replace=False));n.extend(rng.choice(ni,k,replace=False))
 cand=np.r_[m,n];labels=np.r_[np.ones(len(m),int),np.zeros(len(n),int)]
 for cond in ['C1','C2']:
  for treat in ['D0','D0C','D2']:
   w,eps,cliprate,sizes=train_fed(cond,treat)
   pt=expit(z[test]@w);pc=expit(z[cal]@w)
   iso=IsotonicRegression(out_of_bounds='clip').fit(pc,y[cal])
   pm=expit(z[cand]@w);pr=expit(z[cand]@wr)
   a,tpr=attack(-logloss(pm,y[cand]),labels)
   ra,rtpr=attack(logloss(pr,y[cand])-logloss(pm,y[cand]),labels)
   rows.append(dict(seed=SEED,condition=cond,treatment=treat,clip_threshold=CLIP,
     test_auc=roc_auc_score(y[test],pt),brier=brier_score_loss(y[test],pt),
     isotonic_brier=brier_score_loss(y[test],iso.predict(pt)),clip_fraction=cliprate,
     matched_attack_auc=a,matched_tpr_1pct=tpr,
     reference_gap_attack_auc=ra,reference_gap_tpr_1pct=rtpr,
     conditional_epsilon=eps if eps is not None else float('inf'),delta=DELTA))
   print('FINISHED',seed,cond,treat,'AUC',round(rows[-1]['test_auc'],5),'clip',round(cliprate,4),'epsilon',eps,flush=True)
out=pd.DataFrame(rows)
display(out)
print('MEANS:',out.groupby(['condition','treatment']).mean(numeric_only=True).to_string())
out.to_csv('/content/paper3_three_seed_50round_validation.csv',index=False)
print('No end-to-end DP claim. Reference-fitted scaler is not a public preprocessing mechanism. Conditional accountant assumes correct Poisson sampling and per-client mechanism; source Paper III is not reproduced.')
